# 🏙️ NeuralCity Applied GenAI Intern Assignment
## AI-Powered Agricultural Data Q&A System

**Candidate:** *(Your Name)*  
**Date:** *(Submission Date)*  
**Model Used:** `gemini-2.5-flash`  
**Dataset:** [Crop Production Statistics — India (Kaggle)](https://www.kaggle.com/datasets/abhinand05/crop-production-in-india)

---

### 📋 Assignment Overview

This notebook builds an end-to-end **natural language Q&A system** over agricultural data using:
- **Gemini 2.5 Flash** — structured JSON intent extraction (no `exec()`)
- **Pandas** — deterministic, auditable data operations
- **Matplotlib** — chart generation
- **Gradio** — interactive UI with `share=True`

The system supports **6 question types**, maintains full **provenance/auditability**, and includes a formal **evaluation section** with 8 test questions plus manual verification.


## 📐 Requirement-to-Implementation Mapping

| Requirement | Implementation |
|---|---|
| Gemini 2.5 Flash | `google-generativeai`, model=`gemini-2.5-flash` |
| No `exec()` | Structured JSON intent → predefined Pandas functions |
| 6 question types | `top_state`, `trend_comparison`, `top_n`, `aggregate_total`, `district_level`, `out_of_scope` |
| Provenance / Auditability | Every answer includes `intent`, `filters`, `rows_used`, `source_col` metadata |
| Pandas | All data ops via named functions; no dynamic code eval |
| Matplotlib | `generate_chart()` produces bar/line charts per intent |
| Gradio UI | `gr.Interface` with `share=True` |
| Evaluation Section | 8 curated questions, expected vs actual, pass/fail |
| Manual Verification | Markdown cell with spot-check instructions |
| Design Note Submission | Checklists at end of notebook |


In [112]:
# ── Install Dependencies ──────────────────────────────────────────────────────
!pip install -q google-generativeai gradio pandas matplotlib
print("✅ All dependencies installed.")

✅ All dependencies installed.


## 📊 Dataset Information

**Source:** Kaggle — *Crop Production in India*  
**Link:** https://www.kaggle.com/datasets/abhinand05/crop-production-in-india

**Columns:**
| Column | Description |
|---|---|
| `State_Name` | Indian state |
| `District_Name` | District within state |
| `Crop_Year` | Year of production |
| `Season` | Kharif / Rabi / Whole Year / etc. |
| `Crop` | Crop name |
| `Area` | Area under cultivation (hectares) |
| `Production` | Total production (tonnes) |

### ⬆️ How to Upload the Dataset

**Before running the next cell**, upload `crop_production.csv` using ONE of these methods:

**Method A — Colab File Panel (easiest):**
1. Click the 📁 folder icon in the left sidebar
2. Click the ⬆️ upload icon
3. Select `crop_production.csv` from your computer
4. Wait for upload to complete, then run the cell below

**Method B — Code upload widget:**
```python
from google.colab import files
files.upload()   # run this in a new cell, then upload crop_production.csv
```

> ⚠️ **The notebook requires the real dataset.** No synthetic fallback is used.
> The file must be named exactly `crop_production.csv`.

In [113]:
# ── Load & Clean Data (Real Dataset Only) ───────────────────────────────────
import os, warnings
import pandas as pd

warnings.filterwarnings("ignore")

# ── Step 1: Locate the uploaded CSV ──────────────────────────────────────────
# We check two common Colab paths:
#   /content/crop_production.csv  — default Colab upload destination
#   crop_production.csv           — current working directory
SEARCH_PATHS = [
    "/content/crop_production.csv",
    "crop_production.csv",
]

csv_path = None
for p in SEARCH_PATHS:
    if os.path.exists(p):
        csv_path = p
        break

# ── Step 2: Hard-stop with a clear message if file is missing ────────────────
if csv_path is None:
    raise FileNotFoundError(
        "\n"
        "❌  crop_production.csv not found.\n"
        "\n"
        "Please upload the real dataset before running this cell:\n"
        "  Method A: Click the 📁 folder icon → ⬆️ upload → select crop_production.csv\n"
        "  Method B: Run in a new cell:\n"
        "            from google.colab import files; files.upload()\n"
        "\n"
        "Download the dataset from:\n"
        "  https://www.kaggle.com/datasets/abhinand05/crop-production-in-india\n"
    )

# ── Step 3: Load CSV into a Pandas DataFrame ─────────────────────────────────
print(f"📂 Loading dataset from: {csv_path}")
df = pd.read_csv(csv_path)

# ── Step 4: Standardise column names (strip whitespace, replace spaces) ──────
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# ── Step 5: Validate that all required columns are present ───────────────────
REQUIRED_COLS = ["State_Name", "District_Name", "Crop_Year", "Season", "Crop", "Area", "Production"]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(
        f"\n❌  Missing required columns: {missing}\n"
        f"    Found columns: {df.columns.tolist()}\n"
        "    Ensure you uploaded the correct crop_production.csv file."
    )

# ── Step 6: Cast numeric columns; coerce bad values to NaN ──────────────────
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")
df["Area"]       = pd.to_numeric(df["Area"],       errors="coerce")
df["Crop_Year"]  = pd.to_numeric(df["Crop_Year"],  errors="coerce").astype("Int64")

# ── Step 7: Drop rows where critical numeric fields are missing ───────────────
df = df.dropna(subset=["Production", "Area", "Crop_Year"])

# ── Step 8: Strip whitespace from string columns ──────────────────────────────
for col in ["State_Name", "District_Name", "Season", "Crop"]:
    if col in df.columns:
        df[col] = df[col].str.strip()

# ── Step 9: Summary stats to confirm the dataset looks correct ───────────────
print(f"✅ Dataset loaded successfully: {len(df):,} rows × {len(df.columns)} columns")
print(f"   States   : {df['State_Name'].nunique()} unique")
print(f"   Districts: {df['District_Name'].nunique()} unique")
print(f"   Crops    : {df['Crop'].nunique()} unique")
print(f"   Years    : {int(df['Crop_Year'].min())} – {int(df['Crop_Year'].max())}")
print(f"   Seasons  : {df['Season'].unique().tolist()}")
print()
df.head(5)

📂 Loading dataset from: /content/crop_production.csv
✅ Dataset loaded successfully: 242,361 rows × 7 columns
   States   : 33 unique
   Districts: 646 unique
   Crops    : 124 unique
   Years    : 1997 – 2015
   Seasons  : ['Kharif', 'Whole Year', 'Autumn', 'Rabi', 'Summer', 'Winter']



,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [114]:
# ── Configure Gemini 2.0 Flash ────────────────────────────────────────────────
import google.generativeai as genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-2.0-flash"
model = genai.GenerativeModel(MODEL_NAME)
print(f"✅ Gemini configured — model: {MODEL_NAME}")


✅ Gemini configured — model: gemini-2.0-flash


In [115]:
# ── Intent Extraction (Structured JSON via Gemini) ────────────────────────────

import json
import re
import time

def extract_intent(question: str) -> dict:
    """
    Uses Gemini to convert a natural language question into
    a structured intent JSON.
    """

    prompt = f"""
You are an intent classifier for an Indian agricultural dataset.

Return ONLY valid JSON.

Question: {question}

Possible intents:
- top_state
- trend_comparison
- top_n
- aggregate_total
- district_level
- out_of_scope

Examples:

Question: Which state produced the most rice in 2015?
{{
  "intent":"top_state",
  "metric":"Production",
  "crop":"rice",
  "state":null,
  "district":null,
  "year":2015,
  "year_start":null,
  "year_end":null,
  "season":null,
  "n":null,
  "aggregation":"max",
  "confidence":1.0,
  "reasoning":"Highest production state query."
}}

Question: Show wheat production trend from 2000 to 2020.
{{
  "intent":"trend_comparison",
  "metric":"Production",
  "crop":"wheat",
  "state":null,
  "district":null,
  "year":null,
  "year_start":2000,
  "year_end":2020,
  "season":null,
  "n":null,
  "aggregation":"sum",
  "confidence":1.0,
  "reasoning":"Trend over years."
}}

Question: What are the top 5 states by sugarcane production?
{{
  "intent":"top_n",
  "metric":"Production",
  "crop":"sugarcane",
  "state":null,
  "district":null,
  "year":null,
  "year_start":null,
  "year_end":null,
  "season":null,
  "n":5,
  "aggregation":"max",
  "confidence":1.0,
  "reasoning":"Top N ranking query."
}}

Question: What is the total cotton production in Maharashtra?
{{
  "intent":"aggregate_total",
  "metric":"Production",
  "crop":"cotton",
  "state":"Maharashtra",
  "district":null,
  "year":null,
  "year_start":null,
  "year_end":null,
  "season":null,
  "n":null,
  "aggregation":"sum",
  "confidence":1.0,
  "reasoning":"Aggregate production query."
}}

Question: Which district in Punjab has the highest production area?
{{
  "intent":"district_level",
  "metric":"Area",
  "crop":null,
  "state":"Punjab",
  "district":null,
  "year":null,
  "year_start":null,
  "year_end":null,
  "season":null,
  "n":null,
  "aggregation":"max",
  "confidence":1.0,
  "reasoning":"District-level query."
}}

Question: What is the GDP of India in 2023?
{{
  "intent":"out_of_scope",
  "metric":"Production",
  "crop":null,
  "state":null,
  "district":null,
  "year":2023,
  "year_start":null,
  "year_end":null,
  "season":null,
  "n":null,
  "aggregation":"sum",
  "confidence":1.0,
  "reasoning":"Not answerable from crop dataset."
}}

Return JSON only.
"""

    defaults = {
        "intent": "out_of_scope",
        "metric": "Production",
        "crop": None,
        "state": None,
        "district": None,
        "year": None,
        "year_start": None,
        "year_end": None,
        "season": None,
        "n": None,
        "aggregation": "sum",
        "confidence": 0.0,
        "reasoning": ""
    }

    try:

        response = None

        for attempt in range(5):
            try:
                time.sleep(2)
                response = model.generate_content(prompt)
                break

            except Exception as api_error:

                if attempt < 4:
                    print(
                        f"Retry {attempt+1}/5 "
                        f"after API error: {str(api_error)[:120]}"
                    )
                    time.sleep(8)
                else:
                    raise

        raw = response.text.strip()

        raw = re.sub(
            r"^```json\s*|^```\s*|```$",
            "",
            raw,
            flags=re.MULTILINE
        ).strip()

        intent = json.loads(raw)

        for k, v in defaults.items():
            intent.setdefault(k, v)

        return intent

    except json.JSONDecodeError as e:

        print("JSON Parse Error:", e)

        result = defaults.copy()
        result["intent"] = "unknown"
        result["reasoning"] = f"JSON parse error: {e}"

        return result

    except Exception as e:

        print("Gemini Error:", e)

        result = defaults.copy()
        result["intent"] = "unknown"
        result["reasoning"] = str(e)

        return result


# ── Quick Tests ───────────────────────────────────────────────

print("TEST 1")
print(json.dumps(
    extract_intent("Which state produced the most rice in 2015?"),
    indent=2
))

print("\nTEST 2")
print(json.dumps(
    extract_intent("What are the top 5 states by sugarcane production?"),
    indent=2
))

print("\nTEST 3")
print(json.dumps(
    extract_intent("Show wheat production trend from 2000 to 2020"),
    indent=2
))

TEST 1


Retry 1/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 2/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 3/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 4/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


...

Gemini Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 43.382388653s.
{
  "intent": "unknown",
  "metric": "Production",
  "crop": null,
  "state": null,
  "district": null,
  "year": null,
  "year_start": null,
  "year_end": null,
  "seaso

Retry 1/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 2/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 3/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 4/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc
Gemini Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 59.03299227s.
{
  "intent": "unknown"

In [122]:
print(json.dumps(
    extract_intent("Which state produced the most rice in 2015?"),
    indent=2
))

Retry 1/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 2/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 3/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 4/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Gemini Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 34.117933759s.
{
  "intent": "unknown",
  "metric": "Production",
  "crop": null,
  "state": null,
  "district": null,
  "year": null,
  "year_start": null,
  "year_end": null,
  "seaso

In [116]:
# ── Predefined Pandas Functions (no exec()) ───────────────────────────────────

def _filter_df(intent: dict) -> tuple[pd.DataFrame, dict]:
    """Apply crop/state/district/season/year filters. Returns (filtered_df, provenance)."""
    fdf = df.copy()
    filters_applied = {}

    if intent.get("crop"):
        fdf = fdf[fdf["Crop"].str.lower() == intent["crop"].lower()]
        filters_applied["crop"] = intent["crop"]
    if intent.get("state"):
        fdf = fdf[fdf["State_Name"].str.lower() == intent["state"].lower()]
        filters_applied["state"] = intent["state"]
    if intent.get("district"):
        fdf = fdf[fdf["District_Name"].str.lower() == intent["district"].lower()]
        filters_applied["district"] = intent["district"]
    if intent.get("season"):
        fdf = fdf[fdf["Season"].str.lower() == intent["season"].lower()]
        filters_applied["season"] = intent["season"]
    if intent.get("year"):
        fdf = fdf[fdf["Crop_Year"] == intent["year"]]
        filters_applied["year"] = intent["year"]
    if intent.get("year_start") and intent.get("year_end"):
        fdf = fdf[(fdf["Crop_Year"] >= intent["year_start"]) &
                  (fdf["Crop_Year"] <= intent["year_end"])]
        filters_applied["year_range"] = f"{intent['year_start']}–{intent['year_end']}"

    return fdf, filters_applied


def handle_top_state(intent: dict) -> dict:
    fdf, filters = _filter_df(intent)
    metric = intent.get("metric", "Production")
    if metric not in fdf.columns:
        metric = "Production"
    agg = fdf.groupby("State_Name")[metric].sum().sort_values(ascending=False)
    top = agg.index[0]
    val = agg.iloc[0]
    return {
        "answer": f"**{top}** leads with {val:,.0f} tonnes of {metric.lower()}.",
        "data": agg.head(10).reset_index().rename(columns={"State_Name": "State", metric: metric}),
        "chart_type": "bar",
        "x": "State", "y": metric,
        "title": f"Top States by {metric}",
        "provenance": {"intent": "top_state", "filters": filters,
                       "rows_used": len(fdf), "source_col": metric},
    }


def handle_trend_comparison(intent: dict) -> dict:
    fdf, filters = _filter_df(intent)
    metric = intent.get("metric", "Production")
    if metric not in fdf.columns:
        metric = "Production"
    trend = fdf.groupby("Crop_Year")[metric].sum().reset_index()
    trend.columns = ["Year", metric]
    first, last = trend.iloc[0], trend.iloc[-1]
    change = ((last[metric] - first[metric]) / first[metric] * 100) if first[metric] else 0
    direction = "increased" if change >= 0 else "decreased"
    return {
        "answer": (f"{metric} {direction} by **{abs(change):.1f}%** "
                   f"from {int(first['Year'])} to {int(last['Year'])}."),
        "data": trend,
        "chart_type": "line",
        "x": "Year", "y": metric,
        "title": f"{metric} Trend Over Years",
        "provenance": {"intent": "trend_comparison", "filters": filters,
                       "rows_used": len(fdf), "source_col": metric},
    }


def handle_top_n(intent: dict) -> dict:
    fdf, filters = _filter_df(intent)
    n = int(intent.get("n") or 5)
    metric = intent.get("metric", "Production")
    if metric not in fdf.columns:
        metric = "Production"
    agg = fdf.groupby("State_Name")[metric].sum().nlargest(n).reset_index()
    agg.columns = ["State", metric]
    return {
        "answer": f"Top {n} states by {metric}: " + ", ".join(agg["State"].tolist()),
        "data": agg,
        "chart_type": "bar",
        "x": "State", "y": metric,
        "title": f"Top {n} States by {metric}",
        "provenance": {"intent": "top_n", "filters": filters,
                       "rows_used": len(fdf), "source_col": metric},
    }


def handle_aggregate_total(intent: dict) -> dict:
    fdf, filters = _filter_df(intent)
    metric = intent.get("metric", "Production")
    if metric not in fdf.columns:
        metric = "Production"
    agg_func = intent.get("aggregation", "sum")
    val = getattr(fdf[metric], agg_func)()
    label = {"sum": "Total", "mean": "Average", "max": "Maximum", "min": "Minimum"}.get(agg_func, agg_func.title())
    summary = fdf.groupby("State_Name")[metric].sum().reset_index().sort_values(metric, ascending=False)
    summary.columns = ["State", metric]
    return {
        "answer": f"{label} {metric}: **{val:,.0f}** tonnes (across {len(fdf):,} records).",
        "data": summary.head(10),
        "chart_type": "bar",
        "x": "State", "y": metric,
        "title": f"{label} {metric} by State",
        "provenance": {"intent": "aggregate_total", "filters": filters,
                       "rows_used": len(fdf), "source_col": metric,
                       "aggregation": agg_func},
    }


def handle_district_level(intent: dict) -> dict:
    fdf, filters = _filter_df(intent)
    metric = intent.get("metric", "Production")
    if metric not in fdf.columns:
        metric = "Production"
    agg = fdf.groupby("District_Name")[metric].sum().sort_values(ascending=False).reset_index()
    agg.columns = ["District", metric]
    top = agg.iloc[0] if len(agg) else None
    answer = (f"Top district: **{top['District']}** with {top[metric]:,.0f} tonnes."
              if top is not None else "No district data found for these filters.")
    return {
        "answer": answer,
        "data": agg.head(10),
        "chart_type": "bar",
        "x": "District", "y": metric,
        "title": f"District-level {metric}",
        "provenance": {"intent": "district_level", "filters": filters,
                       "rows_used": len(fdf), "source_col": metric},
    }


def handle_out_of_scope(intent: dict) -> dict:
    return {
        "answer": ("⚠️ This question is outside the scope of the agricultural dataset. "
                   "Please ask about crop production, area, trends, or district/state comparisons."),
        "data": pd.DataFrame(),
        "chart_type": None,
        "provenance": {"intent": "out_of_scope", "reasoning": intent.get("reasoning", "")},
    }


INTENT_HANDLERS = {
    "top_state":        handle_top_state,
    "trend_comparison": handle_trend_comparison,
    "top_n":            handle_top_n,
    "aggregate_total":  handle_aggregate_total,
    "district_level":   handle_district_level,
    "out_of_scope":     handle_out_of_scope,
}

print("✅ All 6 intent handlers registered:")
for k in INTENT_HANDLERS:
    print(f"   • {k}")


✅ All 6 intent handlers registered:
   • top_state
   • trend_comparison
   • top_n
   • aggregate_total
   • district_level
   • out_of_scope


In [117]:
# ── Answer Generation Pipeline ────────────────────────────────────────────────

def answer_question(question: str) -> dict:
    """
    Full pipeline:
      1. Extract intent via Gemini (structured JSON)
      2. Route to appropriate Pandas handler
      3. Return answer + data + provenance
    """
    intent = extract_intent(question)
    intent_type = intent.get("intent", "out_of_scope")
    handler = INTENT_HANDLERS.get(intent_type, handle_out_of_scope)
    result = handler(intent)
    result["question"] = question
    result["intent_parsed"] = intent
    return result

# ── Quick pipeline test ───────────────────────────────────────────────────────
_sample = answer_question("Which state produced the most wheat in 2010?")
print("Q:", _sample["question"])
print("A:", _sample["answer"])
print("Provenance:", json.dumps(_sample["provenance"], indent=2))


Retry 1/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 2/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 3/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 4/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Gemini Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 14.817408703s.
Q: Which state produced the most wheat in 2010?
A: ⚠️ This question is outside the scope of the agricultural dataset. Please ask about crop production, area, trends, or d

In [118]:
# ── Chart Generation ──────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import io, base64

CHART_STYLE = {
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#2d3047",
    "text.color":       "#e8eaf0",
    "xtick.color":      "#9ca3af",
    "ytick.color":      "#9ca3af",
    "axes.labelcolor":  "#c9cdd8",
    "grid.color":       "#2d3047",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
}
plt.rcParams.update(CHART_STYLE)

GRADIENT_COLORS = ["#4f8ef7", "#7c5cbf", "#3ecf8e", "#f5a623", "#e74c3c",
                   "#1abc9c", "#9b59b6", "#e67e22", "#2ecc71", "#3498db"]

def generate_chart(result: dict) -> str | None:
    """
    Generates a Matplotlib chart from result dict.
    Returns base64-encoded PNG string, or None if no chart applicable.
    """
    if not result.get("chart_type") or result["data"].empty:
        return None

    data       = result["data"]
    chart_type = result["chart_type"]
    x_col      = result.get("x", data.columns[0])
    y_col      = result.get("y", data.columns[1])
    title      = result.get("title", "")

    fig, ax = plt.subplots(figsize=(10, 5))

    if chart_type == "bar":
        bars = ax.bar(data[x_col].astype(str), data[y_col],
                      color=GRADIENT_COLORS[:len(data)], edgecolor="#0f1117",
                      linewidth=0.8, zorder=3)
        ax.bar_label(bars, fmt="{:,.0f}", padding=4,
                     color="#c9cdd8", fontsize=8, fontweight="bold")
        ax.set_xticks(range(len(data)))
        ax.set_xticklabels(data[x_col].astype(str), rotation=35,
                           ha="right", fontsize=9)

    elif chart_type == "line":
        ax.fill_between(data[x_col], data[y_col],
                        alpha=0.15, color="#4f8ef7")
        ax.plot(data[x_col], data[y_col],
                color="#4f8ef7", linewidth=2.5,
                marker="o", markersize=5, markerfacecolor="#0f1117",
                markeredgecolor="#4f8ef7", markeredgewidth=2, zorder=3)

    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v:,.0f}"))
    ax.set_title(title, fontsize=14, fontweight="bold",
                 color="#e8eaf0", pad=14)
    ax.set_xlabel(x_col, fontsize=10, labelpad=8)
    ax.set_ylabel(y_col + " (tonnes)", fontsize=10, labelpad=8)
    ax.grid(axis="y", zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# Test chart
_chart_b64 = generate_chart(_sample)
if _chart_b64:
    from IPython.display import Image, display
    display(Image(data=base64.b64decode(_chart_b64)))
    print("✅ Chart generated successfully.")
else:
    print("No chart for this result (out_of_scope or empty data).")


No chart for this result (out_of_scope or empty data).


In [119]:
# ── Gradio UI ─────────────────────────────────────────────────────────────────
import gradio as gr
from IPython.display import Image as IPImage
import base64, json, io
from PIL import Image as PILImage

EXAMPLE_QUESTIONS = [
    "Which state produced the most rice in 2015?",
    "Show me the trend of wheat production from 2000 to 2020.",
    "What are the top 5 states by sugarcane production?",
    "What is the total cotton production in Maharashtra?",
    "Which district in Punjab has the highest wheat yield?",
    "What is the average temperature in India?",  # out of scope
]

def gradio_respond(question: str):
    """Gradio handler — returns (answer_text, chart_image, provenance_json)."""
    if not question.strip():
        return "Please enter a question.", None, "{}"

    result = answer_question(question)
    answer = result["answer"]

    # Convert b64 chart → PIL image for Gradio
    chart_img = None
    chart_b64 = generate_chart(result)
    if chart_b64:
        chart_img = PILImage.open(io.BytesIO(base64.b64decode(chart_b64)))

    provenance = json.dumps(result.get("provenance", {}), indent=2)
    return answer, chart_img, provenance


with gr.Blocks(
    title="🌾 AgriInsight — NeuralCity GenAI Demo",
    theme=gr.themes.Base(
        primary_hue="blue",
        secondary_hue="purple",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
    ).set(
        body_background_fill="#0f1117",
        block_background_fill="#1a1d27",
        input_background_fill="#12151f",
        block_border_width="1px",
        block_border_color="#2d3047",
        button_primary_background_fill="#4f8ef7",
        button_primary_text_color="#ffffff",
    ),
) as demo:

    gr.Markdown("""
# 🌾 AgriInsight — India Crop Q&A
### Powered by Gemini 2.5 Flash · NeuralCity Applied GenAI Assignment

Ask natural language questions about **Indian crop production data** (2000–2021).
""")

    with gr.Row():
        with gr.Column(scale=3):
            question_box = gr.Textbox(
                label="Your Question",
                placeholder="e.g. Which state produced the most rice in 2015?",
                lines=2,
                elem_id="q_box",
            )
            with gr.Row():
                submit_btn = gr.Button("🔍 Ask", variant="primary", scale=2)
                clear_btn  = gr.Button("🗑️ Clear", scale=1)

            gr.Examples(
                examples=EXAMPLE_QUESTIONS,
                inputs=question_box,
                label="💡 Try these examples",
            )

        with gr.Column(scale=1):
            gr.Markdown("### 📊 Supported Question Types")
            gr.Markdown("""
- `top_state` — Best state for a crop
- `trend_comparison` — Year-over-year trends
- `top_n` — Ranked lists
- `aggregate_total` — Totals & averages
- `district_level` — District breakdown
- `out_of_scope` — Graceful rejection
""")

    gr.Markdown("---")

    with gr.Row():
        with gr.Column(scale=2):
            answer_box = gr.Markdown(label="📝 Answer")
            chart_box  = gr.Image(label="📈 Chart", type="pil", height=380)
        with gr.Column(scale=1):
            provenance_box = gr.Code(
                label="🔍 Provenance / Audit Trail",
                language="json",
                lines=15,
            )

    # Wire up
    submit_btn.click(
        fn=gradio_respond,
        inputs=question_box,
        outputs=[answer_box, chart_box, provenance_box],
    )
    clear_btn.click(
        fn=lambda: ("", None, "{}"),
        outputs=[answer_box, chart_box, provenance_box],
    )
    question_box.submit(
        fn=gradio_respond,
        inputs=question_box,
        outputs=[answer_box, chart_box, provenance_box],
    )

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b3ae8db97288f0bb02.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🧪 Evaluation Section

8 curated questions covering all intent types. Run the cell below to auto-evaluate.

| # | Question | Expected Intent | Expected Keywords in Answer |
|---|---|---|---|
| 1 | Which state produced the most rice in 2015? | `top_state` | State name, number |
| 2 | Show wheat production trend from 2000 to 2020 | `trend_comparison` | % change, direction |
| 3 | Top 5 states by sugarcane production | `top_n` | 5 state names |
| 4 | Total cotton production in Maharashtra | `aggregate_total` | Total, tonnes |
| 5 | Which district in Punjab has highest area? | `district_level` | District name |
| 6 | Average maize production in Kharif season | `aggregate_total` | Average, tonnes |
| 7 | How did soybean production change 2005–2015? | `trend_comparison` | % change |
| 8 | What is the GDP of India? | `out_of_scope` | outside the scope |


In [120]:
# ── Automated Evaluation ──────────────────────────────────────────────────────
import pandas as pd, json

EVAL_QUESTIONS = [
    {
        "id": 1,
        "question": "Which state produced the most rice in 2015?",
        "expected_intent": "top_state",
        "keywords": ["tonnes"],
    },
    {
        "id": 2,
        "question": "Show wheat production trend from 2000 to 2020",
        "expected_intent": "trend_comparison",
        "keywords": ["%", "increased", "decreased"],
    },
    {
        "id": 3,
        "question": "What are the top 5 states by sugarcane production?",
        "expected_intent": "top_n",
        "keywords": [],   # must have 5 state names — checked by rows
    },
    {
        "id": 4,
        "question": "What is the total cotton production in Maharashtra?",
        "expected_intent": "aggregate_total",
        "keywords": ["tonnes"],
    },
    {
        "id": 5,
        "question": "Which district in Punjab has the highest production area?",
        "expected_intent": "district_level",
        "keywords": ["district", "tonnes"],
    },
    {
        "id": 6,
        "question": "What is the average maize production in Kharif season?",
        "expected_intent": "aggregate_total",
        "keywords": ["tonnes"],
    },
    {
        "id": 7,
        "question": "How did soybean production change between 2005 and 2015?",
        "expected_intent": "trend_comparison",
        "keywords": ["%"],
    },
    {
        "id": 8,
        "question": "What is the GDP of India in 2023?",
        "expected_intent": "out_of_scope",
        "keywords": ["scope", "outside", "⚠️"],
    },
]

eval_results = []
print("Running evaluation — this may take ~30s (Gemini API calls)…\n")

for q in EVAL_QUESTIONS:
    result = answer_question(q["question"])
    got_intent   = result["intent_parsed"].get("intent", "unknown")
    intent_ok    = got_intent == q["expected_intent"]
    answer_lower = result["answer"].lower()
    kw_ok        = all(kw.lower() in answer_lower for kw in q["keywords"]) if q["keywords"] else True
    rows_used    = result.get("provenance", {}).get("rows_used", "N/A")
    passed       = intent_ok and kw_ok

    eval_results.append({
        "ID":              q["id"],
        "Question":        q["question"][:55] + "…" if len(q["question"]) > 55 else q["question"],
        "Expected Intent": q["expected_intent"],
        "Got Intent":      got_intent,
        "Intent ✓":        "✅" if intent_ok else "❌",
        "Keywords ✓":      "✅" if kw_ok else "❌",
        "Rows Used":       rows_used,
        "PASS/FAIL":       "✅ PASS" if passed else "❌ FAIL",
    })
    status = "✅" if passed else "❌"
    print(f"  {status}  Q{q['id']}: {q['question'][:60]}")
    print(f"      Answer: {result['answer'][:100]}")
    print()

eval_df = pd.DataFrame(eval_results)
total   = len(eval_df)
passed  = (eval_df["PASS/FAIL"] == "✅ PASS").sum()
print(f"\n{'='*60}")
print(f"  EVALUATION SUMMARY: {passed}/{total} passed ({passed/total*100:.0f}%)")
print(f"{'='*60}\n")
eval_df


Running evaluation — this may take ~30s (Gemini API calls)…



Retry 1/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 2/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 3/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 4/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


...

Retry 2/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 3/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 4/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Gemini Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 20.760261544s.
  ❌  Q8: What is the GDP of India in 2023?
      Answer: ⚠️ This question is outside the scope of the agricultural dataset. Please ask about crop production,


  EVALUATI

,ID,Question,Expected Intent,Got Intent,Intent ✓,Keywords ✓,Rows Used,PASS/FAIL
0,1,Which state produced the most rice in 2015?,top_state,unknown,❌,❌,N/A,❌ FAIL
1,2,Show wheat production trend from 2000 to 2020,trend_comparison,unknown,❌,❌,N/A,❌ FAIL
2,3,What are the top 5 states by sugarcane product...,top_n,unknown,❌,✅,N/A,❌ FAIL
3,4,What is the total cotton production in Maharas...,aggregate_total,unknown,❌,❌,N/A,❌ FAIL
4,5,Which district in Punjab has the highest produ...,district_level,unknown,❌,❌,N/A,❌ FAIL
5,6,What is the average maize production in Kharif...,aggregate_total,unknown,❌,❌,N/A,❌ FAIL
6,7,How did soybean production change between 2005...,trend_comparison,unknown,❌,❌,N/A,❌ FAIL
7,8,What is the GDP of India in 2023?,out_of_scope,unknown,❌,✅,N/A,❌ FAIL


## 🔬 Manual Verification

### How to spot-check an answer

1. **Pick any answer** from the evaluation above and note its `rows_used` and `filters` from the provenance JSON.
2. **Run the cell below** to replicate the computation manually with raw Pandas.
3. **Compare** the manually computed value to the system's answer — they should match exactly.

This confirms that no dynamic code execution (`exec()`) is involved — only deterministic, auditable Pandas operations.


In [121]:
# ── Manual Verification Cell ──────────────────────────────────────────────────
# Change these values to verify any question from the evaluation.

VERIFY_QUESTION = "Which state produced the most rice in 2015?"

print(f"Verifying: '{VERIFY_QUESTION}'")
print("─" * 60)

# System answer
sys_result = answer_question(VERIFY_QUESTION)
print(f"System answer  : {sys_result['answer']}")
print(f"Provenance     : {json.dumps(sys_result['provenance'], indent=2)}")

# Manual replication
print("\n── Manual Pandas replication ──")
manual = (
    df[
        (df["Crop"].str.lower() == "rice") &
        (df["Crop_Year"] == 2015)
    ]
    .groupby("State_Name")["Production"]
    .sum()
    .sort_values(ascending=False)
)
print(manual.head(5).to_string())
print(f"\n✅ Manual top state: {manual.index[0]}  ({manual.iloc[0]:,.0f} tonnes)")
print("    Compare to system answer above — values should match.")


Verifying: 'Which state produced the most rice in 2015?'
────────────────────────────────────────────────────────────


Retry 1/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 2/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 3/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Retry 4/5 after API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-enc


Gemini Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 36.595176018s.
System answer  : ⚠️ This question is outside the scope of the agricultural dataset. Please ask about crop production, area, trends, or district/state comparisons.
Provena

## 📋 Design Note & Submission Checklists

### 🏗️ Architecture & Design Notes

| Decision | Rationale |
|---|---|
| **Structured JSON intent** | Avoids `exec()` entirely; all data ops via named, auditable Python functions |
| **Gemini 2.5 Flash** | Low latency, strong instruction following for JSON extraction |
| **6 intent types** | Covers the full range of analytical questions over tabular agricultural data |
| **Provenance dict** | Every answer carries `filters`, `rows_used`, `source_col`, `aggregation` for full audit |
| **Fallback to `out_of_scope`** | Graceful degradation — never crashes, always explains why it can't answer |
| **Real dataset only** | Requires `crop_production.csv` upload; no synthetic fallback ensures submission uses genuine data |
| **Dark-themed Gradio UI** | Consistent with data dashboard conventions; `share=True` enables public links |

---

### ✅ Pre-Submission Checklist

#### Code Quality
- [ ] All cells run top-to-bottom without errors
- [ ] No `exec()`, `eval()`, or `subprocess` calls
- [ ] All Gemini calls wrapped in try/except
- [ ] Intent JSON validated before routing

#### Functionality
- [ ] All 6 intent types produce correct answers on sample questions
- [ ] Charts render for bar and line types
- [ ] `out_of_scope` returns a friendly message (not a crash)
- [ ] Provenance dict populated for every answer

#### Evaluation
- [ ] All 8 evaluation questions run and results displayed in DataFrame
- [ ] Pass rate ≥ 75% (6/8)
- [ ] Manual verification cell confirms Pandas replication matches system answer

#### UI
- [ ] Gradio launches without error
- [ ] `share=True` generates a public URL (printed in output)
- [ ] Example questions are clickable
- [ ] Provenance JSON visible in sidebar

#### Submission Artifacts
- [ ] `.ipynb` file saved with all outputs
- [ ] Gradio public URL noted below
- [ ] `crop_production.csv` uploaded and loaded (real dataset confirmed in cell output)
- [ ] Evaluation screenshot/export attached

---

### 📎 Submission Details

| Field | Value |
|---|---|
| **Candidate Name** | *(fill in)* |
| **Submission Date** | *(fill in)* |
| **Gradio Public URL** | *(paste URL from demo.launch() output)* |
| **Dataset Source** | https://www.kaggle.com/datasets/abhinand05/crop-production-in-india |
| **Model Used** | `gemini-2.5-flash` |
| **Evaluation Score** | *(e.g. 7/8 — 87.5%)* |
| **Known Limitations** | *(e.g. district queries require exact name matching)* |
